# <center>**IROS**<center>

**Libraries**

In [1]:
import numpy as np
import mbloodmoon.iros_management as iros
import mbloodmoon as bm

**IROS**

In [2]:
root_path = "/mnt/d/PhD_AASS/Coding/Images_fits/"
mask_file = root_path + "wfm_mask.fits"
simul_data = root_path + "iros_simulation_GC_LMC/20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb/"

cam_a = "cam1a"
cam_b = "cam1b"
dataset = "reconstructed"

filepaths = bm.simulation_files(simul_data)
wfm = bm.codedmask(mask_file, upscale_x=5, upscale_y=1)
sdlA = bm.simulation(filepaths[cam_a][dataset])
sdlB = bm.simulation(filepaths[cam_b][dataset])

max_iterations = 25
snr_threshold = 5

n_test = 0

In [ ]:
iros_output_name: str = root_path + f"iros_output{n_test}.fits"

try:
    iros_output = iros.load_iros_output(iros_output_name)
    
except FileNotFoundError:
    from mbloodmoon.images import upscale, compose
    iros_output, residuals = iros.perform_iros(
        camerasID=(cam_a, cam_b),
        camera=wfm,
        sdl_camA=sdlA,
        sdl_camB=sdlB,
        max_iterations=max_iterations,
        snr_threshold=snr_threshold,
        dataset=dataset,
    )
    
    iros.save_iros_output(iros_output, mask_file, iros_output_name)

    names = tuple(root_path + f"skyres_IROS_{cam.upper()}{n_test}.fits" for cam in (cam_a, cam_b))
    comp_name = root_path + f"composed_skyres_IROS_{cam_a.upper()}{cam_b.upper()}{n_test}.fits"
    sdls = (sdlA, sdlB)
    detectors = tuple(bm.count(wfm, sdl.data)[0] for sdl in sdls)
    variances = tuple(bm.variance(wfm, d) for d in detectors)
    snrs = tuple(bm.snratio(sky, np.clip(var_, a_min=1, a_max=None)) for sky, var_ in zip(residuals, variances))

    ups_skies = tuple(upscale(sky, upscale_y=8) for sky in residuals)
    ups_snrs = tuple(upscale(snr, upscale_y=8) for snr in snrs)

    for res, snr, sdl, name in zip(ups_skies, ups_snrs, sdls, names):
        iros.save_sky(res, snr, sdl, name)
    
    iros.save_sky(
        sky=compose(*ups_skies, strict=False)[0],
        snr=compose(*ups_snrs, strict=False)[0],
        sdl=sdlA,
        save_to=comp_name,
    )

# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!


**Compute Parameters**

In [4]:
iros_data_name = root_path + f"iros_data{n_test}.fits"

try:
    iros_data = iros.load_iros_data(iros_data_name)

except FileNotFoundError:
    log = iros.gen_params_log((cam_a, cam_b))

    iros_data = iros.compute_params(
        iros_output=iros_output,
        camera=wfm,
        sdl_camA=sdlA,
        sdl_camB=sdlB,
        log=log,
    )

    # WARNING: the px position in this DB might not match with upscaled skies
    iros.save_iros_data(
        data=iros_data,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=iros_data_name,
    )

# Loading data...
# Loading completed!


**Catalog Comparison**

In [3]:
dataset_name = root_path + f"database{n_test}.fits"
names = tuple(root_path + f"skyres_IROS_{cam.upper()}{n_test}.fits" for cam in (cam_a, cam_b))
comp_name = root_path + f"composed_skyres_IROS_{cam_a.upper()}{cam_b.upper()}{n_test}.fits"

try:
    dataset = iros.load_iros_data(dataset_name)
    resA, res_snrA = iros.load_sky(names[0])
    resB, res_snrB = iros.load_sky(names[1])
    comp_res, comp_snr = iros.load_sky(comp_name)

except FileNotFoundError:
    catalogA = filepaths[cam_a]["sources"]
    catalogB = filepaths[cam_b]["sources"]
    
    # WARNING: source assignment relies only on catalog sources
    dataset = iros.compare_w_catalog(
        data=iros_data,
        catalogA=catalogA,
        catalogB=catalogB,
        camerasID=(cam_a, cam_b),
        min_flux=0.1,
    )

    iros.save_iros_data(
        data=dataset,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=dataset_name,
    )

# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!


**Plot for Skies and SNRs**

In [4]:
from mbloodmoon.images import upscale, compose

skyA = iros.make_sky(
    data=dataset,
    cameraID=cam_a,
    camera=wfm,
    background=None,
)

skyB = iros.make_sky(
    data=dataset,
    cameraID=cam_b,
    camera=wfm,
    background=None,
)

skyA_wbg = upscale(skyA, upscale_y=8) + resA
skyB_wbg = upscale(skyB, upscale_y=8) + resB

In [5]:
from mbloodmoon.io import SimulationDataLoader
from mbloodmoon.mask import CodedMaskCamera
from mbloodmoon.mask import count, variance, snratio

def get_significance(
    skymap: np.array,
    sdl: SimulationDataLoader,
    camera: CodedMaskCamera,
) -> np.array:
    """Computes SNR for the given sky."""
    detector, _ = count(camera, sdl.data)
    var = variance(camera, detector)
    return snratio(skymap, np.clip(var, a_min=1, a_max=None))

snrA = get_significance(skyA, sdlA, wfm)
snrB = get_significance(skyB, sdlB, wfm)

snrA_wbg = upscale(snrA, upscale_y=8) + res_snrA
snrB_wbg = upscale(snrB, upscale_y=8) + res_snrB

In [6]:
names = tuple(root_path + f"sky_IROS_{cam.upper()}{n_test}.fits" for cam in (cam_a, cam_b))
iros.save_sky(skyA_wbg, snrA_wbg, sdlA, save_to=names[0])
iros.save_sky(skyB_wbg, snrB_wbg, sdlB, save_to=names[1])

skycomp, _ = compose(skyA_wbg, skyB_wbg, strict=False)
snrcomp, _ = compose(snrA_wbg, snrB_wbg, strict=False)
iros.save_sky(skycomp, snrcomp, sdlA, save_to=root_path + f"skycomp_IROS_{cam_a.upper()}{cam_b.upper()}{n_test}.fits")

# Saving Sky...


 [astropy.io.fits.verify]


# Saving completed!
# Saving Sky...


# Saving completed!
# Saving Sky...
# Saving completed!


**Now the PLOTZ**

In [8]:
skyA_wbg_plot = skyA_wbg.copy(); skyA_wbg_plot[skyA_wbg < 0] = 0

sources_posA = [
    (8*y, x) for y, x in zip(
        dataset["cam1a"]["y"]["data"],
        dataset["cam1a"]["x"]["data"],
    )
]

iros.plot_sky(
    sky=skyA_wbg_plot,
    title=f"Galactic Center {cam_a.upper()}",
    sources_ID=dataset["cam1a"]["catalog_name"]["data"],
    sources_pos=sources_posA,
    highlight_pos=True,
)

: 

In [ ]:
skyB_wbg_plot = skyB_wbg.copy(); skyB_wbg_plot[skyB_wbg < 0] = 0

sources_posB = [
    (8*y, x) for y, x in zip(
        dataset["cam1b"]["y"]["data"],
        dataset["cam1b"]["x"]["data"],
    )
]

iros.plot_sky(
    sky=skyB_wbg_plot,
    title=f"Galactic Center {cam_b.upper()}",
    sources_ID=dataset["cam1b"]["catalog_name"]["data"],
    sources_pos=sources_posB,
    highlight_pos=True,
)

In [ ]:
skycomp_plot = skycomp.copy(); skycomp_plot[skycomp < 0] = 0

sources_posA = [
    (8*y, x) for y, x in zip(
        dataset["cam1a"]["y"]["data"],
        dataset["cam1a"]["x"]["data"],
    )
]

iros.plot_sky(
    sky=skycomp,
    title=f"Galactic Center - {cam_a.upper()}-{cam_b.upper()} composition",
    sources_ID=dataset["cam1a"]["catalog_name"]["data"],
    sources_pos=sources_posA,
    highlight_pos=False,
)

In [ ]:
snrA_wbg_plot = snrA_wbg.copy(); snrA_wbg_plot[snrA_wbg < 0] = 0

iros.plot_sky(
    sky=snrA_wbg_plot,
    title=f"Galactic Center SNR {cam_a.upper()}",
    sources_ID=dataset["cam1a"]["catalog_name"]["data"],
    sources_pos=sources_posA,
    highlight_pos=True,
)

: 

In [ ]:
snrB_wbg_plot = snrB_wbg.copy(); snrB_wbg_plot[snrB_wbg < 0] = 0

iros.plot_sky(
    sky=snrB_wbg_plot,
    title=f"Galactic Center SNR {cam_b.upper()}",
    sources_ID=dataset["cam1b"]["catalog_name"]["data"],
    sources_pos=sources_posB,
    highlight_pos=True,
)

In [ ]:
snrcomp_plot = snrcomp.copy(); snrcomp_plot[snrcomp < 0] = 0

sources_posA = [
    (8*y, x) for y, x in zip(
        dataset["cam1a"]["y"]["data"],
        dataset["cam1a"]["x"]["data"],
    )
]

iros.plot_sky(
    sky=snrcomp_plot,
    title=f"Galactic Center - {cam_a.upper()}-{cam_b.upper()} composition",
    sources_ID=dataset["cam1a"]["catalog_name"]["data"],
    sources_pos=sources_posA,
    highlight_pos=False,
)

<br>

<br>

<br>

**Let's check the Statistics of the Counts**